In [ ]:
# ── Cell 1: Install & Imports ──────────────────────────────────────────────────
!pip install ultralytics --quiet

import torch
import os, random, shutil
from ultralytics import YOLO

print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
# ── Cell 2: Dataset Setup ──────────────────────────────────────────────────────
TRAIN_IMG_SRC = "/kaggle/input/datasets/ailienpham/taco-balanced/train/images"
TRAIN_LBL_SRC = "/kaggle/input/datasets/ailienpham/taco-balanced/train/labels"

NEW_TRAIN_IMG = "/kaggle/working/taco/train/images"
NEW_TRAIN_LBL = "/kaggle/working/taco/train/labels"
VAL_IMG       = "/kaggle/working/taco/val/images"
VAL_LBL       = "/kaggle/working/taco/val/labels"

for d in [NEW_TRAIN_IMG, NEW_TRAIN_LBL, VAL_IMG, VAL_LBL]:
    os.makedirs(d, exist_ok=True)

CLASS_NAMES = [
    'Aluminium foil', 'Bottle cap', 'Bottle', 'Broken glass', 'Can',
    'Carton', 'Cigarette', 'Cup', 'Lid', 'Other litter',
    'Other plastic', 'Paper', 'Plastic bag - wrapper',
    'Plastic container', 'Pop tab', 'Straw', 'Styrofoam piece', 'Unlabeled litter'
]
HARD_CLASSES     = {3, 6, 10, 17}
OVERSAMPLE_TIMES = 3


def classes_in_label(lbl_path):
    found = set()
    if os.path.exists(lbl_path):
        with open(lbl_path) as f:
            for line in f:
                p = line.strip().split()
                if p:
                    found.add(int(p[0]))
    return found


def copy_pair(img_name, dest_img_dir, dest_lbl_dir, suffix=""):
    stem, ext = os.path.splitext(img_name)
    shutil.copy(
        os.path.join(TRAIN_IMG_SRC, img_name),
        os.path.join(dest_img_dir, stem + suffix + ext)
    )
    lbl_src = os.path.join(TRAIN_LBL_SRC, stem + ".txt")
    if os.path.exists(lbl_src):
        shutil.copy(lbl_src, os.path.join(dest_lbl_dir, stem + suffix + ".txt"))


all_images = sorted([f for f in os.listdir(TRAIN_IMG_SRC) if f.endswith(".jpg")])
random.seed(42)
random.shuffle(all_images)

split      = int(0.8 * len(all_images))
train_imgs = all_images[:split]
val_imgs   = all_images[split:]

for img in val_imgs:
    stem = os.path.splitext(img)[0]
    shutil.copy(os.path.join(TRAIN_IMG_SRC, img), os.path.join(VAL_IMG, img))
    lbl = os.path.join(TRAIN_LBL_SRC, stem + ".txt")
    if os.path.exists(lbl):
        shutil.copy(lbl, os.path.join(VAL_LBL, stem + ".txt"))

oversampled = 0
for img in train_imgs:
    copy_pair(img, NEW_TRAIN_IMG, NEW_TRAIN_LBL)
    stem = os.path.splitext(img)[0]
    lbl_path = os.path.join(TRAIN_LBL_SRC, stem + ".txt")
    if classes_in_label(lbl_path) & HARD_CLASSES:
        for k in range(1, OVERSAMPLE_TIMES):
            copy_pair(img, NEW_TRAIN_IMG, NEW_TRAIN_LBL, suffix=f"_aug{k}")
            oversampled += 1

print(f"Train images : {len(os.listdir(NEW_TRAIN_IMG))} (incl. {oversampled} oversampled)")
print(f"Val   images : {len(os.listdir(VAL_IMG))}")

In [ ]:
# ── Cell 3: Write YAML ─────────────────────────────────────────────────────────
yaml_content = f"""path: /kaggle/working/taco
train: train/images
val:   val/images

nc: {len(CLASS_NAMES)}
names: {CLASS_NAMES}
"""

with open("/kaggle/working/taco.yaml", "w") as f:
    f.write(yaml_content)

print("taco.yaml written!")

In [ ]:
# ── Cell 4: Train Baseline YOLOv8s ────────────────────────────────────────────
yolo_baseline = YOLO("yolov8s.pt")

results_baseline = yolo_baseline.train(
    data="/kaggle/working/taco.yaml",

    epochs=100,
    imgsz=640,
    batch=16,
    device=0,

    optimizer="AdamW",
    lr0=1e-3,
    lrf=0.01,
    weight_decay=5e-4,
    momentum=0.937,
    cos_lr=True,
    warmup_epochs=3,
    warmup_momentum=0.8,
    warmup_bias_lr=0.1,

    mosaic=1.0,
    mixup=0.15,
    copy_paste=0.4,
    close_mosaic=15,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    degrees=10.0, translate=0.1, scale=0.5, shear=2.0,
    perspective=0.0005, flipud=0.1, fliplr=0.5,

    patience=20,
    seed=42,
    verbose=True,
    project="/kaggle/working/runs/detect",
    name="baseline_100epochs"
)

print("\nBaseline training complete!")
print(f"  mAP50    : {results_baseline.results_dict.get('metrics/mAP50(B)', 'N/A')}")
print(f"  mAP50-95 : {results_baseline.results_dict.get('metrics/mAP50-95(B)', 'N/A')}")

In [ ]:
import shutil, base64
from IPython.display import HTML

shutil.make_archive('/kaggle/working/baseline_100epochs', 'zip',
                    '/kaggle/working/runs/detect/baseline_100epochs')

with open('/kaggle/working/baseline_100epochs.zip', 'rb') as f:
    data = base64.b64encode(f.read()).decode()

HTML(f'<a href="data:application/zip;base64,{data}" download="baseline_100epochs.zip">⬇️ Click here to download baseline results</a>')